In [1]:
import sys
import json
import joblib
from time import time
from collections import defaultdict
from itertools import product
# TO CHANGE
BASEDIR = "../../.."
sys.path.insert(0, BASEDIR)

In [2]:
from pprint import pprint

In [3]:
from src.kg_model import KnowledgeGraphModelConfig, KnowledgeGraphModel
from src.utils.data_structs import QueryInfo

from src.pipelines.qa.kg_reasoning.medium_reasoner.entities_extractor import EntitiesExtractorConfig, EntitiesExtractor
from src.pipelines.qa.kg_reasoning.medium_reasoner.entities2nodes_matching import Entities2NodesMatcherConfig, Entities2NodesMatcher

from src.pipelines.qa.kg_reasoning.weak_reasoner.knowledge_retriever import KnowledgeRetrieverConfig, KnowledgeRetriever
from src.pipelines.qa.kg_reasoning.weak_reasoner.knowledge_retriever.traversal_methods import WaterCirclesRetriever, WaterCirclesSearchConfig, \
    MixturedTripletsRetriever, MixturedGraphSearchConfig, AStarTripletsRetriever, AStarGraphSearchConfig, \
        NaiveBFSTripletsRetriever, NaiveBFSGraphSearchConfig, NaiveTripletsRetriever, NaiveGraphSearchConfig, \
            BeamSearchTripletsRetriever, GraphBeamSearchConfig


/home/dzigen/Desktop/Projects/PersonalAI/.pai_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Preparing testings configs

In [ ]:
AVAILABLE_RETRIEVER_CONFIGS = {
    'naive_retriever': KnowledgeRetrieverConfig(
        retriever_method='naive_retriever',
        retriever_config=NaiveGraphSearchConfig(),
        filter_method=None,
        filter_config=None
    ),
    # 'astar': KnowledgeRetrieverConfig(
    #     retriever_method='astar',
    #     retriever_config=AStarGraphSearchConfig(),
    #     filter_method=None,
    #     filter_config=None
    # ),
    'beamsearch': KnowledgeRetrieverConfig(
        retriever_method='beamsearch',
        retriever_config=GraphBeamSearchConfig(),
        filter_method=None,
        filter_config=None
    ),
    'watercircles': KnowledgeRetrieverConfig(
        retriever_method='watercircles',
        retriever_config=WaterCirclesSearchConfig(),
        filter_method=None,
        filter_config=None
    ),
    # 'naive_bfs': KnowledgeRetrieverConfig(
    #     retriever_method='naive_bfs',
    #     retriever_config=NaiveBFSGraphSearchConfig(),
    #     filter_method=None,
    #     filter_config=None
    # ),
    # 'mixture (beamsearch + astart)': KnowledgeRetrieverConfig(
    #     retriever_method='mixture',
    #     retriever_config=MixturedGraphSearchConfig(
    #         retriever1_name='beamsearch',
    #         retriever1_config=GraphBeamSearchConfig(),
    #         retriever2_name='astar',
    #         retriever2_config=AStarGraphSearchConfig()
    #     ),
    #     filter_method=None,
    #     filter_config=None
    # ),
    'mixture (beamsearch + watercircles)': KnowledgeRetrieverConfig(
        retriever_method='mixture',
        retriever_config=MixturedGraphSearchConfig(
            retriever1_name='beamsearch',
            retriever1_config=GraphBeamSearchConfig(),
            retriever2_name='watercircles',
            retriever2_config=WaterCirclesSearchConfig()
        ),
        filter_method=None,
        filter_config=None
    ),
    'mixture (beamsearch + naive_retriever)': KnowledgeRetrieverConfig(
        retriever_method='mixture',
        retriever_config=MixturedGraphSearchConfig(
            retriever1_name='beamsearch',
            retriever1_config=GraphBeamSearchConfig(),
            retriever2_name='naive_retriever',
            retriever2_config=NaiveGraphSearchConfig()
        ),
        filter_method=None,
        filter_config=None
    )
}

BASE_KGMODEL_CONFIGS_DIR='./kg_configs'
AVAILABLE_KGMODEL_CONFIGS = {
    'hotpotqa': f"{BASE_KGMODEL_CONFIGS_DIR}/hotpotqa_qwen257b_230126_v2prompts_kgconfig",
    'triviaqa': f"{BASE_KGMODEL_CONFIGS_DIR}/triviaqa_qwen257b_290126_v2prompts_kgconfig",
    'diaasq': f"{BASE_KGMODEL_CONFIGS_DIR}/diaasq_qwen257b_260126_v2prompts_kgconfig",
    'natural_questions': f"{BASE_KGMODEL_CONFIGS_DIR}/naturalqa_qwen257b_020226_v2prompts_kgconfig",
    'musique': f"{BASE_KGMODEL_CONFIGS_DIR}/musique_qwen257b_050226_v2prompts_kgconfig",
    '2wiki': f"{BASE_KGMODEL_CONFIGS_DIR}/2wiki_qwen257b_080226_v2prompts_kgconfig"
}

AVAILABLE_DATASET_QUESTIONS = {
    'hotpotqa': [
        "Were Scott Derrickson and Ed Wood of the same nationality?",
        "What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?",
        "What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?"
    ],
    'triviaqa': [
        "Who had an 80s No 1 hit with Hold On To The Nights?",
        "Men Against the Sea and Pitcairn's Island were two sequels to what famous novel?",
        "Kim Carnes' nine weeks at No 1 with Bette Davis Eyes was interrupted for one week by which song?"
    ],  
    'diaasq': [
        "Which device is better in battery life: iPhone11 Pro Max or Xiaomi 11?",
        "The majority of speakers have positive, neutral or negative sentiment about connection of Apple?",
        "The majority of speakers have positive, neutral or negative sentiment about signal of Apple?"
    ],
    'natural_questions': [
        "who sang what in the world's come over you",
        "who produces the most wool in the world",
        "where does alaska the last frontier take place"
    ],
    'musique': [
        "Who is the spouse of the Green performer?",
        "Who founded the company that distributed the film UHF?",
        "What administrative territorial entity is the owner of Ciudad Deportiva located?"
    ],
    '2wiki': [
        "Who is the mother of the director of film Polish-Russian War (Film)?",
        "Which film came out first, Blind Shaft or The Mask Of Fu Manchu?",
        "When did John V, Prince Of Anhalt-Zerbst's father die?"
    ]
}

#### Start performance testing of kg-traversal algorithms

In [5]:
SELECTED_DATASET = '2wiki' # TO CHANGE

In [6]:
kg_config = joblib.load(AVAILABLE_KGMODEL_CONFIGS[SELECTED_DATASET])

In [7]:
#kg_config.embedders_configs['default'].device = 'cpu'
kg_config.agents_configs['default'].agent_config.credentials['port'] = 11439

In [14]:
pprint(kg_config)

KnowledgeGraphModelConfig(log=<src.utils.logger.Logger object at 0x7f8a10b766b0>,
                          verbose=False,
                          graph_struct_config=GraphModelConfig(log=<src.utils.logger.Logger object at 0x7f8a10b76710>,
                                                               verbose=False,
                                                               driver_config=GraphDriverConfig(db_vendor='neo4j',
                                                                                               db_config=GraphDBConnectionConfig(db_info={'db': 'personalaigraphdb',
                                                                                                                                          'table': 'personalaigraphtable'},
                                                                                                                                 params={'pwd': 'password',
                                                                         

In [9]:
kg_model = KnowledgeGraphModel(kg_config)

In [10]:
kg_model.count_items()

{'graph_info': {'triplets': 301483, 'nodes': 94425},
 'embeddings_info': {'nodes': 94425, 'triplets': 84163},
 'nodestree_info': None}

In [15]:
entities_extractor_config = EntitiesExtractorConfig(lang='en')
entities_extractor = EntitiesExtractor(
    kg_model.AVAILABLE_AGENTS[kg_model.AGENTS_MAP.qa_pipeline], 
    entities_extractor_config
)

entities2nodes_matcher_config = Entities2NodesMatcherConfig()
entities2nodes_matcher = Entities2NodesMatcher(kg_model, entities2nodes_matcher_config)

In [16]:
ELAPSED_TIME = defaultdict(list)

In [17]:
for retriever_name, retriever_config in AVAILABLE_RETRIEVER_CONFIGS.items():
    print("="*20)
    print("Retriever: ", retriever_name)
    print("="*20)
    knowledge_retriever=KnowledgeRetriever(kg_model, retriever_config)
    for question in AVAILABLE_DATASET_QUESTIONS[SELECTED_DATASET]:
        print("Question: ", question)
        entities, _, _ = entities_extractor.perform(question)
        matched_kg_objects, _, _ = entities2nodes_matcher.perform(entities)

        base_entities = sorted(list(filter(lambda entity: len(matched_kg_objects[entity]) > 0, matched_kg_objects.keys())))
        objects_groups = list(product(*[matched_kg_objects[k] for k in base_entities]))
        spec_group = objects_groups[0]
        formated_objects_group = list(map(lambda item: item.text, spec_group))

        query_info = QueryInfo(
            query=question, 
            entities=base_entities, 
            linked_nodes=list(spec_group),
            linked_nodes_by_entities=list(map(lambda pair: [base_entities[pair[0]], pair[1]], enumerate(formated_objects_group)))
        )
        
        print("* extracted entities: ", entities)
        print("* linked nodes: ", formated_objects_group)

        s_time = time()
        _, _, _ = knowledge_retriever.retrieve(query_info)
        e_time = time()
        cur_elapsed_time = round(e_time-s_time, 5)
        print("Elapsed time: ", cur_elapsed_time)
        print()

        ELAPSED_TIME[retriever_name].append(cur_elapsed_time)

Retriever:  naive_retriever
Question:  Who is the mother of the director of film Polish-Russian War (Film)?
* extracted entities:  ['mother', 'director', 'Polish-Russian War', 'Film']
* linked nodes:  ['best film', 'russian civil war', 'director', 'mother of']
Elapsed time:  0.8092

Question:  Which film came out first, Blind Shaft or The Mask Of Fu Manchu?
* extracted entities:  ['Blind Shaft', 'The Mask Of Fu Manchu']
* linked nodes:  ['blind shaft', 'the mask of fu manchu']
Elapsed time:  0.76145

Question:  When did John V, Prince Of Anhalt-Zerbst's father die?
* extracted entities:  ['John V', 'Prince Of Anhalt-Zerbst', 'father', 'die']
* linked nodes:  ['john v', 'prince of anhalt-zerbst', 'die hard', 'father']
Elapsed time:  0.91254

Retriever:  beamsearch
Question:  Who is the mother of the director of film Polish-Russian War (Film)?
* extracted entities:  ['mother', 'director', 'Polish-Russian War', 'Film']
* linked nodes:  ['best film', 'russian civil war', 'director', 'mothe

In [19]:
pprint(dict(ELAPSED_TIME))

{'beamsearch': [42.36864, 18.9314, 32.29729],
 'mixture (beamsearch + naive_retriever)': [40.37117, 18.9606, 31.2054],
 'mixture (beamsearch + watercircles)': [51.84112, 22.4739, 39.339],
 'naive_retriever': [0.8092, 0.76145, 0.91254],
 'watercircles': [13.98592, 5.11627, 10.36787]}


#### Performance testing results (log)

-----------------------------

#### <b>10.03.26 | version=2.3.1</b>

##### <b>hotpotqa</b>

ssh port forwarding params:

-L 7503:localhost:7503 -L 7716:localhost:7716 -L 6337:localhost:6337 -L 6438:localhost:6438 -L 27046:localhost:27046 -L 8110:localhost:8110 -L 6408:localhost:6408 -L 5569:localhost:5569 -L 9212:localhost:9212 -L 9610:localhost:9610

results:

* 'beamsearch': [17.07819, 28.466, 27.62225],
* 'mixture (beamsearch + naive_retriever)': [17.39153, 28.45734, 27.70033],
* 'mixture (beamsearch + watercircles)': [21.93259, 38.22448, 37.97074],
* 'naive_retriever': [0.76306, 0.87522, 0.97071],
* 'watercircles': [4.4505, 9.70544, 10.2545]

##### <b>triviaqa</b>

ssh port forwarding params:

-L 7507:localhost:7507 -L 7720:localhost:7720 -L 6341:localhost:6341 -L 6442:localhost:6442 -L 27050:localhost:27050 -L 8114:localhost:8114 -L 6412:localhost:6412 -L 5573:localhost:5573 -L 9216:localhost:9216 -L 9614:localhost:9614

results:

* 'beamsearch': [15.00483, 42.07987, 33.56354]
* 'mixture (beamsearch + naive_retriever)': [14.14912, 39.6168, 32.97866]
* 'mixture (beamsearch + watercircles)': [20.12543, 53.32244, 45.53926]
* 'naive_retriever': [1.46084, 1.10954, 1.01913]
* 'watercircles': [8.2775, 14.47095, 14.41401]

##### <b>diaasq</b>

ssh port forwarding params:

-L 7511:localhost:7511 -L 7724:localhost:7724 -L 6345:localhost:6345 -L 6446:localhost:6446 -L 27054:localhost:27054 -L 8118:localhost:8118 -L 6416:localhost:6416 -L 5577:localhost:5577 -L 9220:localhost:9220 -L 9618:localhost:9618

results:

* 'beamsearch': [23.66457, 20.30537, 22.40761],
* 'mixture (beamsearch + naive_retriever)': [24.32865, 21.41851, 23.57294],
* 'mixture (beamsearch + watercircles)': [32.86905, 32.12405, 34.19856],
* 'naive_retriever': [0.88294, 0.87775, 0.84718],
* 'watercircles': [9.31317, 11.22521, 12.39174]

##### <b>naturalqa</b>

ssh port forwarding params:

-L 7515:localhost:7515 -L 7728:localhost:7728 -L 6349:localhost:6349 -L 6450:localhost:6450 -L 27058:localhost:27058 -L 8122:localhost:8122 -L 6420:localhost:6420 -L 5581:localhost:5581 -L 9224:localhost:9224 -L 9622:localhost:9622

results:

* 'beamsearch': [20.967, 25.6042, 16.54635],
* 'mixture (beamsearch + naive_retriever)': [19.01881, 24.64888, 16.80443],
* 'mixture (beamsearch + watercircles)': [25.68406, 31.49034, 20.28555],
* 'naive_retriever': [1.36257, 0.93043, 0.84973],
* 'watercircles': [8.1765, 7.24532, 5.03049]

##### <b>musique</b>

ssh port forwarding params:

-L 7519:localhost:7519 -L 7732:localhost:7732 -L 6353:localhost:6353 -L 6454:localhost:6454 -L 27062:localhost:27062 -L 8126:localhost:8126 -L 6424:localhost:6424 -L 5585:localhost:5585 -L 9228:localhost:9228 -L 9626:localhost:9626

results:

* 'beamsearch': [20.4905, 40.55799, 12.02317],
* 'mixture (beamsearch + naive_retriever)': [18.91777, 38.5709, 12.18686],
* 'mixture (beamsearch + watercircles)': [25.46522, 49.79102, 16.11667],
* 'naive_retriever': [1.32527, 1.13092, 0.97568],
* 'watercircles': [7.32457, 12.21314, 4.74911]

##### <b>2wiki</b>

ssh port forwarding params:

-L 7523:localhost:7523 -L 7736:localhost:7736 -L 6357:localhost:6357 -L 6458:localhost:6458 -L 27066:localhost:27066 -L 8130:localhost:8130 -L 6428:localhost:6428 -L 5589:localhost:5589 -L 9232:localhost:9232 -L 9630:localhost:9630

results:

* 'beamsearch': [42.36864, 18.9314, 32.29729],
* 'mixture (beamsearch + naive_retriever)': [40.37117, 18.9606, 31.2054],
* 'mixture (beamsearch + watercircles)': [51.84112, 22.4739, 39.339],
* 'naive_retriever': [0.8092, 0.76145, 0.91254],
* 'watercircles': [13.98592, 5.11627, 10.36787]

-----------------------------